In [4]:
using Pkg
using Revise

Pkg.activate("/home/jiaxingl/project/Cersyve.jl/model/double_integrator")
Pkg.status()

using Plots
using LaTeXStrings
using Flux
using JLD2
using Statistics
using Random
using EllipsisNotation
using Printf

include("/home/jiaxingl/project/Cersyve.jl/src/double_integrator.jl");
include("/home/jiaxingl/project/Cersyve.jl/src/model.jl");
# include("../utils.jl");
%%capture

  Activating project at `~/project/Cersyve.jl/model/double_integrator`


Status `~/project/Cersyve.jl/model/double_integrator/Project.toml`
  [da5c29d0] EllipsisNotation v1.8.0
⌅ [587475ba] Flux v0.14.21
  [033835bb] JLD2 v0.5.11
  [b964fa9f] LaTeXStrings v1.4.0
  [91a5bcdd] Plots v1.40.9
  [de0858da] Printf
  [9a3f8284] Random
  [10745b16] Statistics v1.9.0
Info Packages marked with ⌅ have new versions available but compatibility constraints restrict them from upgrading. To see why use `status --outdated`


[ Info: Precompiling Plots [91a5bcdd-55d7-5caf-9e0b-520d859cae80]
ERROR: LoadError: ArgumentError: Platform `riscv64-linux-gnu` is not an officially supported platform
Stacktrace:
 [1] parse(::Type{Base.BinaryPlatforms.Platform}, triplet::String; validate_strict::Bool)
   @ Base.BinaryPlatforms ./binaryplatforms.jl:752
 [2] parse(::Type{Base.BinaryPlatforms.Platform}, triplet::String)
   @ Base.BinaryPlatforms ./binaryplatforms.jl:670
 [3] (::Libiconv_jll.var"#parse_wrapper_platform#1")(x::Any)
   @ Libiconv_jll ~/.julia/packages/JLLWrappers/jXOYx/src/toplevel_generators.jl:113
 [4] (::Libiconv_jll.var"#make_wrapper_dict#2"{Libiconv_jll.var"#parse_wrapper_platform#1"})(dir::Any, x::Any)
   @ Libiconv_jll ~/.julia/packages/JLLWrappers/jXOYx/src/toplevel_generators.jl:145
 [5] top-level scope
   @ ~/.julia/packages/JLLWrappers/jXOYx/src/toplevel_generators.jl:160
 [6] include
   @ ./Base.jl:457 [inlined]
 [7] include_package_for_output(pkg::Base.PkgId, input::String, depot_path::Vector{S

LoadError: Failed to precompile Plots [91a5bcdd-55d7-5caf-9e0b-520d859cae80] to "/home/jiaxingl/.julia/compiled/v1.9/Plots/jl_1dfa3H".

In [3]:
%%capture

Unrecognized magic `%%capture`.

Julia does not use the IPython `%magic` syntax.   To interact with the IJulia kernel, use `IJulia.somefunction(...)`, for example.  Julia macros, string macros, and functions can be used to accomplish most of the other functionalities of IPython magics.


In [2]:
# evaluate value network
seed = 1
Random.seed!(seed)
task = DoubleIntegrator


x_a_low =  [task.x_low; task.u_low]
x_a_high = [task.x_high; task.u_high]
x_a = uniform(x_a_low, x_a_high, 1000000)

# x = uniform(x_low, x_high, 1000000)
pre_affine_Q = create_parallel_affine_Q(task.x_dim, task.u_dim)
ft_affine_Q = create_parallel_affine_Q(task.x_dim, task.u_dim)

Flux.loadmodel!(pre_affine_Q, JLD2.load("Q_pretrain.jld2", "state"))
Flux.loadmodel!(ft_affine_Q, JLD2.load("Q_finetune990.jld2", "state"))

# act_pat = get_activation_pattern(V_model, x)
# act_pat = get_activation_pattern(V_V_prime_model, x)
# log_points, uni_pat_num = count_activation_pattern(act_pat)
# println("Estimated linear segments: $(uni_pat_num[end])")

LoadError: UndefVarError: `Random` not defined

In [ ]:
seed = 1
Random.seed!(seed)

x_a = uniform(x_a_low, x_a_high, 1000000)
x = x_a[1:task.x_dim, :]


T = 100
traj = Array{Float32}(undef, size(x)..., T)
x_temp = copy(x)
for i in 1:T
    traj[.., i] = x_temp
    x_temp = f_pi_model(x_temp)
end

v_true = maximum(h_model(traj)[1, ..], dims=2)[:, 1]
println("Feasible ratio: ", mean(v_true .<= 0))

# Flux.loadmodel!(V_model, JLD2.load("log/repair_20240615_205703/V.jld2", "state"))
# v = V_model(x)[1, :]

v = ft_affine_Q(x_a)[1, :]

ffr = sum((v_true .> 0) .& (v .<= 0)) / sum(v_true .> 0)
tfr = sum((v_true .<= 0) .& (v .<= 0)) / sum(v_true .<= 0)
@printf "False feasible rate: %.4f\n" ffr
@printf "True feasible rate: %.4f" tfr

In [ ]:
# plot experiment figure
_, pre_Q_interval = create_Q_Q_prime(pre_affine_Q, f_pi_model, task)
_, ft_Q_interval = create_Q_Q_prime(ft_affine_Q, f_pi_model, task)

x_1 = Vector{Float32}(range(x_low[1], x_high[1], 100))
x_2 = Vector{Float32}(range(x_low[2], x_high[2], 100))
X_1 = repeat(x_1', length(x_2), 1)
X_2 = repeat(x_2, 1, length(x_1))
X = stack((X_1, X_2), dims=1)

X_with_action = hcat(X, zeros(Float32, size(X, 1), action_dim))


V_pre = pre_Q_interval(X_with_action)[1, ..]
V_rep = ft_Q_interval(X_with_action)[1, ..]


# Flux.loadmodel!(V_model, JLD2.load("log/value_20240604_081619/V.jld2", "state"))
# V_pre = V_model(X)[1, ..]

# Flux.loadmodel!(V_model, JLD2.load("log/repair_20240604_081808/V.jld2", "state"))
# V_rep = V_model(X)[1, ..]




p = contourf(
    x_1, x_2, V_rep,
    levels=10, color=range(HSL(180., 0.4, 1.), stop=HSL(180., 0.4, 0.2), length=10),
    lw=0, alpha=1,
    xlabel=L"x(1)", ylabel=L"x(2)",
    xlimits=(x_low[1], x_high[1]),
    ylimits=(x_low[2], x_high[2]),
    right_margin=5Plots.mm,
)

T = 50
TRAJ = Array{Float32}(undef, size(X)..., T)
X_temp = copy(X)
for i in 1:T
    TRAJ[.., i] = X_temp
    X_temp = f_pi_model(X_temp)
end
V_true = maximum(h_model(TRAJ)[1, ..], dims=3)[.., 1]

contour!(x_1, x_2, V_true, levels=[0], color=HSL(0., 0., 0.1), lw=2)
contour!(x_1, x_2, V_pre, levels=[0], color=HSL(20., 0.9, 0.6), lw=2)
contour!(x_1, x_2, V_rep, levels=[0], color=HSL(240., 0.9, 0.4), lw=2)



seed = 1
Random.seed!(seed)
Flux.loadmodel!(V_model, JLD2.load("log/value_20240604_081619/V.jld2", "state"))

# x = uniform(x_low, x_high, 10000)
# v = V_model(x)[1, ..]
# v_prime = V_model(f_pi_model(x))[1, ..]

x = uniform(x_low, x_high, 10000)
x_with_action = hcat(x, zeros(Float32, size(x, 1), action_dim))
v = ft_Q_interval(x_with_action)[1, ..]

x_prime = f_pi_model(x)
x_prime_with_action = hcat(x_prime, zeros(Float32, size(x_prime, 1), action_dim))
v_prime = V_model(x_prime_with_action)[1, ..]

inv_ce = x[:, (v .<= 0) .& (v_prime .> 0)]
inv_ce_prime = f_pi_model(inv_ce)
scatter!(inv_ce_prime[1, :], inv_ce_prime[2, :], label="", color=HSL(0., 0.7, 0.2), markersize=4)
scatter!(inv_ce[1, :], inv_ce[2, :], label="", color=HSL(0., 0.7, 0.5), markersize=4)

lens = [0.68 0.88; 0.4 0.6]
plot!([lens[1, 1], lens[1, 2], lens[1, 2], lens[1, 1], lens[1, 1]],
    [lens[2, 1], lens[2, 1], lens[2, 2], lens[2, 2], lens[2, 1]],
    color=:black, lw=1, label=false)

plot!(size=(400, 300))
savefig("figure/double_integrator_large.svg")
plot(p)

In [ ]:
# plot experiment figure
p = contourf(
    x_1, x_2, V_rep,
    levels=10, color=range(HSL(180., 0.4, 1.), stop=HSL(180., 0.4, 0.2), length=10),
    lw=0, alpha=1,
    xlabel=L"x(1)", ylabel=L"x(2)",
    right_margin=5Plots.mm,
)

contour!(x_1, x_2, V_true, levels=[0], color=HSL(0., 0., 0.1), lw=2)
contour!(x_1, x_2, V_pre, levels=[0], color=HSL(20., 0.9, 0.6), lw=2)
contour!(x_1, x_2, V_rep, levels=[0], color=HSL(240., 0.9, 0.4), lw=2)

for i in 1:size(inv_ce, 2)
    plot!([inv_ce[1, i], inv_ce_prime[1, i] - 0.002], [inv_ce[2, i], inv_ce_prime[2, i] + 0.002],
        arrow=arrow(:closed), color=HSL(0., 0., 0.2), linewidth=1, label="")
end

scatter!(inv_ce_prime[1, :], inv_ce_prime[2, :], label="", color=HSL(0., 0.7, 0.2), markersize=6)
scatter!(inv_ce[1, :], inv_ce[2, :], label="", color=HSL(0., 0.7, 0.5), markersize=6)

plot!(size=(400, 300), xlimits=(lens[1, 1], lens[1, 2]), ylimits=(lens[2, 1], lens[2, 2]))
savefig("figure/double_integrator_small.svg")
plot(p)

In [ ]:
p = plot([], [], color=HSL(0., 0., 0.1), lw=2, label="True region", legend_columns=-1)
plot!([], [], color=HSL(20., 0.9, 0.6), lw=2, label="Pre-trained region")
scatter!([], [], label="Counterexample", color=HSL(0., 0.7, 0.5))
plot!([], [], color=HSL(240., 0.9, 0.4), lw=2, label="Verified region")
plot!(size=(800, 100), axis=([], false), legendfont=font(10, "Arial"))
savefig("figure/legend.svg")
plot(p)

In [ ]:
# plot heatmap after fine-tuning and feasible region comparison
Flux.loadmodel!(V_model, JLD2.load("log/repair_20240604_081808/V.jld2", "state"))

V_ft = V_model(X)[1, ..]

p = contourf(
    x_1, x_2, V_ft,
    levels=10, color=:balance, lw=0, alpha=0.5,
    xlabel=L"x(1)", ylabel=L"x(2)",
    right_margin=5Plots.mm,
)
contour!(x_1, x_2, V_ft, label="after FT", levels=[0], color=:green1, lw=2)
contour!(x_1, x_2, V, label="before FT", levels=[0], color=:red, lw=2)
contour!(x_1, x_2, V_true, label="ground truth", levels=[0], color=:black, lw=2)

plot!(size=(400, 300))
savefig("figure/after_fine_tuning.pdf")
plot(p)

In [ ]:
# plot introduction figure before fine-tuning
x_1 = Vector{Float32}(range(x_low[1], x_high[1], 100))
x_2 = Vector{Float32}(range(x_low[2], x_high[2], 100))
X_1 = repeat(x_1', length(x_2), 1)
X_2 = repeat(x_2, 1, length(x_1))
X = stack((X_1, X_2), dims=1)

T = 50
TRAJ = Array{Float32}(undef, size(X)..., T)
X_temp = copy(X)
for i in 1:T
    TRAJ[.., i] = X_temp
    X_temp = f_pi_model(X_temp)
end
V_true = maximum(h_model(TRAJ)[1, ..], dims=3)[.., 1]

p = plot([-1, 1, 1, -1, -1], [-1, -1, 1, 1, -1], color=:black, lw=1, label=false)

contour!(x_1, x_2, V_true, levels=[0], color=:gray, lw=2, colorbar=false,
    axis=([], false))

Flux.loadmodel!(V_model, JLD2.load("log/value_20240604_081619/V.jld2", "state"))
V = V_model(X)[1, ..]
contour!(x_1, x_2, V, levels=[0], color=HSL(30., 0.8, 0.5), lw=2)

plot!([], [], color=:gray, lw=2, label=" True region")
plot!([], [], color=HSL(30., 0.8, 0.5), lw=2, label=" Pre-trained region")

plot!(size=(350, 300), legend=:inside, legendfont=font(16, "Arial"))
savefig("figure/intro_before_FT.svg")
plot(p)

In [ ]:
# plot introduction counterexamples
p = plot([-1, 1, 1, -1, -1], [-1, -1, 1, 1, -1], color=:black, lw=1, label=false)

contour!(x_1, x_2, V_true, levels=[0], color=:gray, lw=2, colorbar=false, 
    axis=([], false))

Flux.loadmodel!(V_model, JLD2.load("log/value_20240604_081619/V.jld2", "state"))
V = V_model(X)[1, ..]
contour!(x_1, x_2, V, levels=[0], color=HSL(30., 0.8, 0.5), lw=2)

seed = 1
Random.seed!(seed)
x = uniform(x_low, x_high, 10000)
h = h_model(x)[1, ..]
v = V_model(x)[1, ..]
v_prime = V_model(f_pi_model(x))[1, ..]
con_ce = x[:, (v .<= 0) .& (h .> 0)]
inv_ce = x[:, (v .<= 0) .& (v_prime .> 0)]
scatter!(con_ce[1, :], con_ce[2, :], label="", color=:firebrick2, markersize=4)
scatter!(inv_ce[1, :], inv_ce[2, :], label="Counterexample", color=:firebrick2,
    markersize=4)

plot!(size=(350, 300), legend=:inside, legendfont=font(16, "Arial"))
savefig("figure/intro_counterexample.svg")
plot(p)

In [ ]:
# plot introduction figure after fine-tuning
p = plot([-1, 1, 1, -1, -1], [-1, -1, 1, 1, -1], color=:black, lw=1, label=false)

contour!(x_1, x_2, V_true, levels=[0], color=:gray, lw=2, colorbar=false, 
    axis=([], false))

Flux.loadmodel!(V_model, JLD2.load("log/value_20240604_081619/V.jld2", "state"))
V = V_model(X)[1, ..]
contour!(x_1, x_2, V, levels=[0], color=HSL(30., 0.8, 0.5), lw=2)

Flux.loadmodel!(V_model, JLD2.load("log/repair_20240604_081808/V.jld2", "state"))
V = V_model(X)[1, ..]
contour!(x_1, x_2, V, levels=[0], color=HSL(220., 0.8, 0.5), lw=2)

plot!([-1, 1, 1, -1, -1], [-1, -1, 1, 1, -1], color=:black, lw=1, label=false)

plot!([], [], color=HSL(220., 0.8, 0.5), lw=2, label=" Verified region")

plot!(size=(350, 300), legend=:inside, legendfont=font(16, "Arial"))
savefig("figure/intro_after_FT.svg")
plot(p)